# Baseline Training Notebook — BirdCLEF+ 2026


## 1. Introduction

This notebook trains and compares three baseline classifiers for BirdCLEF+ 2026: a multi-label bioacoustic classification task for 234 species (birds, amphibians, mammals, reptiles, insects) from 5-second audio chunks.

Data source: https://www.kaggle.com/competitions/birdclef-2026/data

Audio files are converted to log-mel spectrograms (128 mel bands, 32 kHz) and classified by one of three CNN backbones (pretrained on ImageNet via `timm`) with a 234-class sigmoid head and binary cross-entropy loss:

| Model | timm ID |
|---|---|
| EfficientNetV2-S | `tf_efficientnetv2_s.in21k_ft_in1k` |
| EfficientNetB0 | `tf_efficientnet_b0.ns_jft_in1k` |
| EfficientNetB3 | `tf_efficientnet_b3.ns_jft_in1k` |

Models are compared on **macro ROC-AUC** and **macro F1** (threshold = 0.5) using 3-fold author-grouped cross-validation on focal recordings combined with a file-grouped 3-fold split of the labeled soundscapes, with the AdamW optimizer, a cosine learning rate schedule, and early stopping on validation macro ROC-AUC.

## 2. Setup

In [ ]:
import os, sys, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import timm
from torch.utils.data import DataLoader, ConcatDataset
from sklearn.metrics import roc_auc_score, f1_score

UTILS = '/kaggle/input/datasets/tsapalyu/birdclef2026-utils' 
COMP  = '/kaggle/input/competitions/birdclef-2026'           

FOCAL_DIR = f'{COMP}/train_audio'
SS_DIR    = f'{COMP}/train_soundscapes'

sys.path.append(UTILS)

from birdclef_utils.dataset import FocalDataset, SoundscapeChunkDataset
from birdclef_utils.constants import NUM_CLASSES

_base = {
    'batch_size':    64,
    'accum_steps':   1,
    'num_workers':   4,
    'lr':            1e-3,
    'weight_decay':  1e-5,
    'max_epochs':    20,
    'patience':      5,
    'num_classes':   NUM_CLASSES,
    'use_amp':       False,
    # loss: 1.0 * BCE + 1.0 * SigmoidFocalLoss (alpha=0.25, gamma=2)
    'bce_weight':    1.0,
    'focal_weight':  1.0,
    'focal_alpha':   0.25,
    'focal_gamma':   2.0,
}

CONFIGS = {
    'effnetv2s': {**_base,
        'model_name': 'tf_efficientnetv2_s.in21k_ft_in1k',
        'model_key':  'effnetv2s',
    },
    'effnetb0': {**_base,
        'model_name': 'tf_efficientnet_b0.ns_jft_in1k',
        'model_key':  'effnetb0',
    },
    'effnetb3': {**_base,
        'model_name': 'tf_efficientnet_b3.ns_jft_in1k',
        'model_key':  'effnetb3',
    },
}

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {torch.cuda.get_device_name(0) if DEVICE=='cuda' else 'CPU'}")
print(f"Models to compare: {list(CONFIGS.keys())}")

## 3 & 4. Data Loading & Exploration and Preprocessing

In [2]:
focal_df = pd.read_csv(f'{UTILS}/train_folds.csv')
ss_df    = pd.read_csv(f'{UTILS}/labeled_soundscapes_split.csv')
with open(f'{UTILS}/label2idx.json') as f:
    label2idx = json.load(f)

print(f'focal samples:      {len(focal_df)}')
print(f'soundscape chunks: {len(ss_df)}')
print(f'num classes:       {NUM_CLASSES}')

def build_loaders(fold, batch_size=32, num_workers=4):
    """Build combined train/val DataLoaders for one fold."""

    # TRAIN: focal + soundscape, everything NOT in this fold ---
    train_focal = FocalDataset(
        focal_df[focal_df['fold'] != fold],
        audio_dir=FOCAL_DIR, label2idx=label2idx, mode='train',
    )
    train_ss = SoundscapeChunkDataset(
        ss_df[ss_df['fold'] != fold],
        audio_dir=SS_DIR, label2idx=label2idx, mode='train',
    )
    train_ds = ConcatDataset([train_focal, train_ss])

    # VAL: focal + soundscape, this fold only ---
    val_focal = FocalDataset(
        focal_df[focal_df['fold'] == fold],
        audio_dir=FOCAL_DIR, label2idx=label2idx, mode='val',
    )
    val_ss = SoundscapeChunkDataset(
        ss_df[ss_df['fold'] == fold],
        audio_dir=SS_DIR, label2idx=label2idx, mode='val',
    )
    val_ds = ConcatDataset([val_focal, val_ss])

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True, drop_last=True,
    )

    # for validation, we can use *2 because During validation there's no gradient computation
    # so each sample uses less GPU memory
    val_loader = DataLoader(
        val_ds, batch_size=batch_size * 1, shuffle=False,
        num_workers=num_workers, pin_memory=True,
    )
    
    # Return val datasets separately too
    # focal-val and soundscape-val independently (if we want to do separate test in case)
    return train_loader, val_loader, val_focal, val_ss


# Load the data
train_loader, val_loader, val_focal, val_ss = build_loaders(fold=0, batch_size=64)

# Sanity check for 1 fold first to verify batch shapes BEFORE building the model
specs, labels = next(iter(train_loader))
print(f"spec batch:  {specs.shape}")
print(f"label batch: {labels.shape}")
print(f"avg species per sample: {labels.sum(dim=1).mean():.2f}")

focal samples:      35549
soundscape chunks: 739
num classes:       234
spec batch:  torch.Size([64, 1, 128, 313])
label batch: torch.Size([64, 234])
avg species per sample: 1.27


## 5. Model Definition

Models are built using a CNN backbone from timm with:
- ImageNet pretrained weights
- Single-channel spectrogram input (`in_chans=1`)
- 234-output multi-label classification head

Three backbones are compared:
- **EfficientNetV2-S** (`tf_efficientnetv2_s.in21k_ft_in1k`) — (as per the paper)
- **EfficientNetB0** (`tf_efficientnet_b0.ns_jft_in1k`) — lighter, faster alternative
- **EfficientNetB3** (`tf_efficientnet_b3.ns_jft_in1k`) — lighter, faster alternative

In [3]:
def build_model(cfg):
    model = timm.create_model(
        cfg['model_name'],
        pretrained=True,
        num_classes=cfg['num_classes'],
        in_chans=1,
    )
    return model.to(DEVICE)

## 6. Training

In [4]:
def train_one_epoch(model, loader, optimizer, criterion, cfg):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad()

    for step, (specs, labels) in enumerate(loader):
        specs  = specs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        # forward
        logits = model(specs)
        loss = criterion(logits, labels)

        # backward (with gradient accumulation)
        (loss / cfg['accum_steps']).backward()
        if (step + 1) % cfg['accum_steps'] == 0:
            optimizer.step()
            optimizer.zero_grad()

        running_loss += loss.item()

    return running_loss / len(loader)


### Metrics

In [ ]:
def macro_roc_auc(y_true, y_pred):
    """Mean per-class ROC-AUC, skipping classes with no positive (or no
    negative) labels — AUC is undefined for those."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    aucs = []
    for c in range(y_true.shape[1]):
        col = y_true[:, c]
        if col.sum() == 0 or col.sum() == len(col):
            continue
        aucs.append(roc_auc_score(col, y_pred[:, c]))
    return float(np.mean(aucs)) if aucs else float('nan')


def macro_f1(y_true, y_pred, threshold=0.5):
    """Macro-averaged F1, restricted to classes that have at least one
    positive label in y_true (undefined classes are skipped, not zeroed)."""
    y_true = np.asarray(y_true)
    y_binary = (np.asarray(y_pred) >= threshold).astype(int)
    valid = y_true.sum(axis=0) > 0
    if not valid.any():
        return float('nan')
    return float(f1_score(y_true[:, valid], y_binary[:, valid],
                          average='macro', zero_division=0))

### Loss Function

Combined `1.0 · BCE + 1.0 · SigmoidFocalLoss`, following the paper (Section 5.2).

Weights and gamma are set in `_base` and can be overridden per model in `CONFIGS`.

In [ ]:
class CombinedBCEFocalLoss(nn.Module):
    """1.0 · BCEWithLogitsLoss + 1.0 · torchvision SigmoidFocalLoss."""
    def __init__(self, bce_weight=1.0, focal_weight=1.0, alpha=0.25, gamma=2.0):
        super().__init__()
        self.bce_weight   = bce_weight
        self.focal_weight = focal_weight
        self.alpha        = alpha
        self.gamma        = gamma
        self.bce          = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        bce_loss   = self.bce(logits, targets)
        focal_loss = torchvision.ops.sigmoid_focal_loss(
            inputs=logits,
            targets=targets,
            alpha=self.alpha,
            gamma=self.gamma,
            reduction='mean',
        )
        return self.bce_weight * bce_loss + self.focal_weight * focal_loss


def build_criterion(cfg):
    return CombinedBCEFocalLoss(
        bce_weight=cfg.get('bce_weight', 1.0),
        focal_weight=cfg.get('focal_weight', 1.0),
        alpha=cfg.get('focal_alpha', 0.25),
        gamma=cfg.get('focal_gamma', 2.0),
    )

### Validation

In [7]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for specs, labels in loader:
        specs = specs.to(DEVICE, non_blocking=True)
        probs = torch.sigmoid(model(specs))
        all_preds.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())
    return np.vstack(all_preds), np.vstack(all_labels)

### Fold Training

In [ ]:
def train_fold(fold, cfg):
    model_key = cfg['model_key']
    print(f"\n{'='*55}\n{model_key.upper()}  —  FOLD {fold}\n{'='*55}")

    train_loader, val_loader, val_focal, val_ss = build_loaders(
        fold, batch_size=cfg['batch_size'], num_workers=cfg['num_workers'],
    )

    print(f"Train samples: {len(train_loader.dataset)}")
    print(f"Val samples:   {len(val_loader.dataset)}")
    print(f"  focal val:      {len(val_focal)}")
    print(f"  soundscape val: {len(val_ss)}")

    model     = build_model(cfg)
    criterion = build_criterion(cfg)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg['max_epochs'],
    )

    ckpt_path = f'/kaggle/working/{model_key}_fold{fold}_last.pth'
    start_epoch = 0
    best_auc = 0.0
    epochs_no_improve = 0

    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        start_epoch       = ckpt['epoch'] + 1
        best_auc          = ckpt['best_auc']
        epochs_no_improve = ckpt['epochs_no_improve']
        print(f"Resuming {model_key} fold {fold} from epoch {start_epoch} "
              f"(best AUC so far {best_auc:.4f})")
    else:
        print(f"No checkpoint found - training {model_key} fold {fold} from scratch.")

    for epoch in range(start_epoch, cfg['max_epochs']):
        t0 = time.time()

        train_loss = train_one_epoch(model, train_loader, optimizer,
                                     criterion, cfg)

        preds, labels = evaluate(model, val_loader)
        val_auc = macro_roc_auc(labels, preds)

        scheduler.step()

        print(f"  epoch {epoch:2d} | loss {train_loss:.4f} | "
              f"val AUC {val_auc:.4f} | {time.time()-t0:.0f}s")

        torch.save({
            'epoch':                epoch,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_auc':             best_auc,
            'epochs_no_improve':    epochs_no_improve,
            'val_auc':              val_auc,
            'model_name':           cfg['model_name'],
            'model_key':            model_key,
            'num_classes':          cfg['num_classes'],
            'in_chans':             1,
        }, f'/kaggle/working/{model_key}_fold{fold}_last.pth')

        if val_auc > best_auc:
            best_auc = val_auc
            epochs_no_improve = 0
            torch.save({'model_state_dict': model.state_dict(),
                        'epoch': epoch, 'val_auc': val_auc},
                       f'/kaggle/working/{model_key}_fold{fold}_best.pth')
            print(f"    -> new best ({best_auc:.4f})")
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= cfg['patience']:
            print(f"  early stopping (best val AUC {best_auc:.4f})")
            break

    return best_auc

In [ ]:
# --- Training control ---
# Set MODEL_KEY to select which backbone to train.
# Set FOLD to select which fold to run.
# Run this cell once per fold per model (or loop over both if compute allows).
SMOKE_TEST = False
MODEL_KEY  = 'effnetv2s'   # 'effnetv2s' | 'effnetb0'
FOLD       = 0              # 0 | 1 | 2

cfg = CONFIGS[MODEL_KEY]

if SMOKE_TEST:
    smoke_cfg = dict(cfg, max_epochs=1)
    print(f"SMOKE TEST: one epoch, fold {FOLD}, model {MODEL_KEY}.")
    train_fold(fold=FOLD, cfg=smoke_cfg)
else:
    best = train_fold(fold=FOLD, cfg=cfg)
    print(f"\n{MODEL_KEY} fold {FOLD} best val AUC: {best:.4f}")

## 7. Evaluation

In [ ]:
def load_fold_model(checkpoint_path, cfg):
    """Rebuild a trained model from a saved checkpoint."""
    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    model = timm.create_model(
        cfg['model_name'],
        pretrained=False,
        num_classes=cfg['num_classes'],
        in_chans=1,
    )
    model.load_state_dict(ckpt['model_state_dict'])
    model.to(DEVICE).eval()
    return model, ckpt

In [ ]:
CHECKPOINT_DIR = '/kaggle/working'
NUM_FOLDS = 3

all_results = {}

for model_key, cfg in CONFIGS.items():
    fold_rows = []
    print(f"\n{'='*60}")
    print(f"Evaluating: {model_key}  ({cfg['model_name']})")
    print(f"{'='*60}")

    for fold in range(NUM_FOLDS):
        ckpt_path = os.path.join(CHECKPOINT_DIR,
                                 f'{model_key}_fold{fold}_best.pth')
        if not os.path.exists(ckpt_path):
            print(f"  Fold {fold}: checkpoint not found, skipping.")
            continue

        print(f"\n  --- Fold {fold} ---")
        model, ckpt = load_fold_model(ckpt_path, cfg)
        print(f"  loaded epoch {ckpt['epoch']}, "
              f"training-time val AUC {ckpt['val_auc']:.4f}")

        _, val_loader, _, _ = build_loaders(
            fold,
            batch_size=cfg['batch_size'],
            num_workers=cfg['num_workers'],
        )

        preds, labels = evaluate(model, val_loader)
        auc = macro_roc_auc(labels, preds)
        f1  = macro_f1(labels, preds)
        print(f"  val AUC: {auc:.4f}  |  val F1 (t=0.5): {f1:.4f}")

        fold_rows.append({
            'fold':    fold,
            'val_AUC': auc,
            'val_F1':  f1,
            'n_val':   len(labels),
        })

        del model
        torch.cuda.empty_cache()

    all_results[model_key] = pd.DataFrame(fold_rows)


# --- Comparison table ---
print("\n" + "=" * 65)
print("Model Comparison — macro ROC-AUC vs macro F1 (threshold = 0.5)")
print("=" * 65)

summary_rows = []
for model_key, df in all_results.items():
    if df.empty:
        continue
    summary_rows.append({
        'model':      model_key,
        'mean_AUC':   df['val_AUC'].mean(),
        'std_AUC':    df['val_AUC'].std(ddof=1),
        'mean_F1':    df['val_F1'].mean(),
        'std_F1':     df['val_F1'].std(ddof=1),
        'folds_done': len(df),
    })

comparison_df = pd.DataFrame(summary_rows)
print(comparison_df.to_string(index=False, float_format='%.4f'))

for model_key, df in all_results.items():
    if not df.empty:
        df.to_csv(f'/kaggle/working/cv_summary_{model_key}.csv', index=False)
comparison_df.to_csv('/kaggle/working/cv_comparison.csv', index=False)
print(f"\nSaved cv_comparison.csv and per-model summaries to /kaggle/working/")

## 8. Comparison & Analysis

In [ ]:
# Per-source (focal vs soundscape) AUC and F1 breakdown, for each model
for model_key, cfg in CONFIGS.items():
    per_source_rows = []
    print(f"\n{'='*60}")
    print(f"Model: {model_key}  —  focal vs soundscape breakdown")
    print(f"{'='*60}")

    for fold in range(3):
        ckpt_path = os.path.join('/kaggle/working',
                                 f'{model_key}_fold{fold}_best.pth')
        if not os.path.exists(ckpt_path):
            print(f"  Fold {fold}: checkpoint not found, skipping.")
            continue

        print(f"\n  --- Fold {fold} (per-source) ---")
        model, ckpt = load_fold_model(ckpt_path, cfg)

        _, _, val_focal, val_ss = build_loaders(
            fold,
            batch_size=cfg['batch_size'],
            num_workers=cfg['num_workers'],
        )
        focal_loader = DataLoader(val_focal, batch_size=cfg['batch_size'],
                                  shuffle=False, num_workers=cfg['num_workers'],
                                  pin_memory=True)
        ss_loader    = DataLoader(val_ss, batch_size=cfg['batch_size'],
                                  shuffle=False, num_workers=cfg['num_workers'],
                                  pin_memory=True)

        fp, fl = evaluate(model, focal_loader)
        sp, sl = evaluate(model, ss_loader)

        focal_auc = macro_roc_auc(fl, fp)
        ss_auc    = macro_roc_auc(sl, sp)
        focal_f1  = macro_f1(fl, fp)
        ss_f1     = macro_f1(sl, sp)
        gap_auc   = focal_auc - ss_auc

        print(f"  focal val      ({len(fl):>5} samples): AUC {focal_auc:.4f}  F1 {focal_f1:.4f}")
        print(f"  soundscape val ({len(sl):>5} samples): AUC {ss_auc:.4f}  F1 {ss_f1:.4f}")
        print(f"  domain gap (AUC):                   {gap_auc:+.4f}")

        per_source_rows.append({
            'fold':           fold,
            'focal_AUC':      focal_auc,
            'focal_F1':       focal_f1,
            'focal_n':        len(fl),
            'soundscape_AUC': ss_auc,
            'soundscape_F1':  ss_f1,
            'soundscape_n':   len(sl),
            'domain_gap_AUC': gap_auc,
        })

        del model
        torch.cuda.empty_cache()

    if per_source_rows:
        df = pd.DataFrame(per_source_rows)
        print(f"\n  Mean focal AUC:      {df['focal_AUC'].mean():.4f} ± {df['focal_AUC'].std(ddof=1):.4f}")
        print(f"  Mean soundscape AUC: {df['soundscape_AUC'].mean():.4f} ± {df['soundscape_AUC'].std(ddof=1):.4f}")
        print(f"  Mean domain gap:     {df['domain_gap_AUC'].mean():+.4f}")
        df.to_csv(f'/kaggle/working/cv_per_source_{model_key}.csv', index=False)
        print(f"  Saved to /kaggle/working/cv_per_source_{model_key}.csv")